# Ποσοτικοποίηση της Κοινής Κληρονομιάς Σχεδίασης σε έναν Στόλο Μετασχηματιστών Ισχύος με την PROC INBREED

## Περίληψη

Οι μετασχηματιστές υποσταθμών ενός διαχειριστή δικτύου σχεδιάζονται σε διαδοχικές γενιές σχεδίασης, με κάθε νέο μοντέλο να προέρχεται από δύο προγενέστερες σχεδιάσεις. Αυτό το notebook αντιμετωπίζει αυτή τη μηχανολογική γενεαλογία ως γενεαλογικό δέντρο (pedigree) και χρησιμοποιεί την **PROC INBREED** για να υπολογίσει συντελεστές ενδογαμίας και προσθετικής συγγένειας (συνδιακύμανσης), ποσοτικοποιώντας πόση κοινή κληρονομιά σχεδίασης μοιράζονται δύο περιουσιακά στοιχεία.

Το γενεαλογικό δέντρο έχει κατασκευαστεί έτσι ώστε η δομή να είναι ερμηνεύσιμη: δύο από τα τέσσερα τρέχοντα μοντέλα του στόλου κατάγονται από ένα ζευγάρι **αδελφών** προγενέστερων σχεδιάσεων και συνεπώς φέρουν συμπυκνωμένη κληρονομιά, ενώ τα άλλα δύο αντλούν από ανεξάρτητους κλάδους. Η διαδικασία το ανακτά με ακρίβεια. Τα δύο μοντέλα που προέρχονται από αδέλφια (`G3_T1`, `G3_T3`) φέρουν το καθένα συντελεστή ενδογαμίας **F = 0,25**· τα άλλα δύο (`G3_T2`, `G3_T4`) έχουν **F = 0**. Ο μέσος συντελεστής ενδογαμίας του στόλου είναι **0,0417**. Διαβάζοντας τον πίνακα προσθετικής συγγένειας, το λιγότερο συγγενικό τρέχον ζεύγος του στόλου είναι **το `G3_T1` & `G3_T3` (συγγένεια = 0)** — δεν μοιράζονται καμία κοινή καταγωγή και αποτελούν τον ισχυρότερο εφεδρικό (N-1) συνδυασμό, κάτι που έχει σημασία επειδή το `G3_T1` είναι από μόνο του ένα από τα πιο ενδογαμικά σχέδια.

## Πηγές Δεδομένων

| Σύνολο Δεδομένων | Γραμμές | Βασικές Μεταβλητές | Περιγραφή |
|---------|------|---------------|-------------|
| `DesignLineage` | 12 | `Generation`, `ID`, `Parent1`, `Parent2`, `Sex` | Ένα μικρό, ντετερμινιστικό μηχανολογικό γενεαλογικό δέντρο σχεδιάσεων μετασχηματιστών υποσταθμών σε τρεις μη επικαλυπτόμενες γενιές σχεδίασης (4 ιδρυτικές πλατφόρμες, 4 παράγωγα δεύτερης γενιάς, 4 τρέχοντα μοντέλα στόλου). Κάθε μη ιδρυτική σχεδίαση καταγράφει τις δύο διακριτές προγενέστερες σχεδιάσεις (`Parent1`, `Parent2`) από τις οποίες προήλθε. Το `Sex` επισημαίνει τον επικεφαλής κλάδο σχεδίασης (M = μηχανολογικός/πυρήνα κλάδος, F = ηλεκτρικός/περιέλιξης κλάδος). Το γενεαλογικό δέντρο είναι χειροκίνητα καθορισμένο — όχι τυχαίο — έτσι ώστε η δομή συγγένειας να είναι γνωστή εκ των προτέρων και να μπορεί να ελεγχθεί έναντι της εξόδου της διαδικασίας. |

# Ποσοτικοποίηση της Κοινής Κληρονομιάς Σχεδίασης σε έναν Στόλο Μετασχηματιστών Ισχύος

## Γιατί μια εταιρεία κοινής ωφέλειας ενδιαφέρεται για την «ενδογαμία»

Ένας διαχειριστής μεταφοράς και διανομής λειτουργεί εκατοντάδες μετασχηματιστές ισχύος υποσταθμών. Για να ελέγξουν το μηχανολογικό κόστος και την προσπάθεια πιστοποίησης, οι κατασκευαστές σπάνια σχεδιάζουν κάθε μετασχηματιστή από το μηδέν — κάθε νέο μοντέλο **κληρονομεί** τη βασική γεωμετρία, την τοπολογία περιέλιξης, τα συστήματα μόνωσης και τις σχεδιάσεις μονωτήρων από ένα ή δύο προγενέστερα μοντέλα. Σε αρκετούς κύκλους προμηθειών αυτό παράγει μια πραγματική *μηχανολογική γενεαλογία*: ένα γενεαλογικό δέντρο σχεδιάσεων.

Αυτή η κοινή κληρονομιά είναι ένας κρυφός κίνδυνος αξιοπιστίας. Αν δύο μετασχηματιστές που προστατεύουν το ίδιο φορτίο (ένα εφεδρικό ζεύγος **N-1**) κατάγονται από την ίδια πρόγονη σχεδίαση, ένα λανθάνον σχεδιαστικό ελάττωμα — μια ιδιοσυχνότητα συντονισμού, ένας μηχανισμός γήρανσης μόνωσης, μια διαδρομή αστοχίας μονωτήρα — είναι πιθανό να υπάρχει και στις **δύο** μονάδες. Μια μοναδική βασική αιτία μπορεί τότε να θέσει εκτός λειτουργίας το εφεδρικό ζεύγος ταυτόχρονα: μια *βλάβη κοινής αιτίας* (common-mode failure).

Η **PROC INBREED** σχεδιάστηκε ακριβώς για να ποσοτικοποιεί αυτού του είδους την κοινή καταγωγή. Παρότι η καταγωγή της βρίσκεται στην κτηνοτροφική και φυτική αναπαραγωγή, τα μαθηματικά της — η αναδρομή προσθετικής συγγένειας του Wright — είναι ανεξάρτητα από τον τομέα εφαρμογής: μετρούν το ποσοστό κληρονομιάς σχεδίασης που μοιράζονται δύο άτομα μέσω κοινών προγόνων. Αντιστοιχίζουμε το λεξιλόγιο της γενετικής στη μηχανολογία περιουσιακών στοιχείων:

| Έννοια INBREED | Ερμηνεία στην εταιρεία κοινής ωφέλειας |
|---|---|
| Άτομο (Individual) | Μια σχεδίαση/μοντέλο μετασχηματιστή |
| Γονέας (sire/dam) | Μια προγενέστερη σχεδίαση από την οποία προήλθε |
| Γενιά (CLASS) | Ένας κύκλος προμηθειών/σχεδίασης |
| Συντελεστής ενδογαμίας *F* | Βαθμός αυτο-όμοιας κληρονομιάς εντός μιας σχεδίασης |
| Συντελεστής συνδιακύμανσης/συγγένειας | Κοινή κληρονομιά σχεδίασης μεταξύ δύο σχεδιάσεων |
| Λιγότερο συγγενικό ζεύγος | Καλύτερος εφεδρικός συνδυασμός για ανθεκτικότητα N-1 |

## Βήμα 1 — Καθορισμός του γενεαλογικού δέντρου σχεδίασης

Εισάγουμε απευθείας το `DesignLineage` ώστε η δομή συγγένειας να είναι σαφής. Υπάρχουν τρεις **μη επικαλυπτόμενες γενιές σχεδίασης**:

- **Γενιά 1** — τέσσερις ιδρυτικές σχεδιάσεις πλατφόρμας (`G1_P1`-`G1_P4`) χωρίς προγόνους (κενοί γονείς).
- **Γενιά 2** — τέσσερις παράγωγες σχεδιάσεις (`G2_D1`-`G2_D4`), καθεμία σχεδιασμένη από δύο **διακριτές** πλατφόρμες Γενιάς 1. Τα `G2_D1` και `G2_D2` είναι *πλήρη αδέλφια* (και τα δύο από `G1_P1` x `G1_P2`)· τα `G2_D3` και `G2_D4` είναι πλήρη αδέλφια (και τα δύο από `G1_P3` x `G1_P4`).
- **Γενιά 3** — τέσσερα τρέχοντα μοντέλα στόλου (`G3_T1`-`G3_T4`). Το `G3_T1` κατασκευάζεται από το ζευγάρι αδελφών `G2_D1` x `G2_D2`, και το `G3_T3` από το ζευγάρι αδελφών `G2_D3` x `G2_D4`· αυτές οι δύο σχεδιάσεις συνεπώς συμπυκνώνουν κληρονομιά. Τα `G3_T2` και `G3_T4` διασταυρώνουν το καθένα τους δύο ανεξάρτητους κλάδους.

Οι στήλες `ID`, `Parent1` και `Parent2` φέρουν το γενεαλογικό δέντρο· το `Sex` καταγράφει ποιος μηχανολογικός κλάδος ηγήθηκε της σχεδίασης. Ένα σύντομο επόμενο βήμα DATA αδειάζει το σύμβολο-θέση `.` ώστε οι ιδρυτικές πλατφόρμες να έχουν κενούς γονείς, όπως αναμένει η PROC INBREED.

In [1]:
ΔΕΔΟΜΕΝΑ DesignLineage;
   LENGTH ID $12 Parent1 $12 Parent2 $12 Sex $1;
   INFILE DATALINES truncover;
   ΕΙΣΟΔΟΣ Generation ID $ Parent1 $ Parent2 $ Sex $;
   DATALINES;
1 G1_P1 . . M
1 G1_P2 . . M
1 G1_P3 . . M
1 G1_P4 . . F
2 G2_D1 G1_P1 G1_P2 M
2 G2_D2 G1_P1 G1_P2 F
2 G2_D3 G1_P3 G1_P4 F
2 G2_D4 G1_P3 G1_P4 M
3 G3_T1 G2_D1 G2_D2 M
3 G3_T2 G2_D1 G2_D3 F
3 G3_T3 G2_D3 G2_D4 F
3 G3_T4 G2_D2 G2_D4 M
;
ΕΚΤΕΛΕΣΗ;

/* Οι ιδρυτικές πλατφόρμες δεν έχουν προγόνους: άδειασμα του συμβόλου-θέσης τελεία */
ΔΕΔΟΜΕΝΑ DesignLineage;
   ΟΡΙΣΜΟΣ DesignLineage;
   ΕΑΝ Parent1 = '.' ΤΟΤΕ Parent1 = ' ';
   ΕΑΝ Parent2 = '.' ΤΟΤΕ Parent2 = ' ';
ΕΚΤΕΛΕΣΗ;

TITLE 'Γενεαλογία Σχεδίασης Μετασχηματιστών';
ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=DesignLineage noobs ΕΤΙΚΕΤΑ;
   ΜΕΤΑΒΛΗΤΗ Generation ID Parent1 Parent2 Sex;
   ΕΤΙΚΕΤΑ Generation='Γενιά' ID='Αναγνωριστικό' Parent1='Γονέας 1' Parent2='Γονέας 2' Sex='Κλάδος (M/F)';
ΕΚΤΕΛΕΣΗ;

                                          Γενεαλογία Σχεδίασης Μετασχηματιστών                                          


     Γενιά               Αναγνωριστικό        Γονέας 1        Γονέας 2        Κλάδος (M/F)
----------  --------------------------  --------------  --------------  ------------------
         1  G1_P1                                                       M
         1  G1_P2                                                       M
         1  G1_P3                                                       M
         1  G1_P4                                                       F
         2  G2_D1                       G1_P1           G1_P2           M
         2  G2_D2                       G1_P1           G1_P2           F
         2  G2_D3                       G1_P3           G1_P4           F
         2  G2_D4                       G1_P3           G1_P4           M
         3  G3_T1                       G2_D1           G2_D2           M
         3  G3_T2            


NOTE: DATA DesignLineage

NOTE: Processing inline DATALINES (12 lines)

NOTE: Read 12 rows from DATALINES.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: DATA DesignLineage


NOTE: Read 12 rows from DesignLineage.
NOTE: Wrote DesignLineage (12 rows, 5 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: Option TITLE changed to Γενεαλογία Σχεδίασης Μετασχηματιστών.
NOTE: PROC PRINT data=DesignLineage

NOTE: PROC PRINT completed: 12 observations printed, 5 variables


## Βήμα 2 — Συντελεστές ενδογαμίας ανά γενιά σχεδίασης

Εκτελούμε την PROC INBREED σε **λειτουργία πολλαπλών γενεών**, ονομάζοντας το `Generation` στη δήλωση `CLASS`, η οποία τυπώνει τη σύνοψη του γενεαλογικού δέντρου ανά γενιά. Η δήλωση `VAR` παρέχει τις τρεις στήλες του γενεαλογικού δέντρου με την απαιτούμενη σειρά: άτομο, πρώτος πρόγονος, δεύτερος πρόγονος.

- Η επιλογή **IND** τυπώνει τον συντελεστή ενδογαμίας *F* κάθε σχεδίασης — ένα μέτρο του πόσο συμπυκνωμένη είναι η δική της κληρονομιά. Μια σχεδίαση κατασκευασμένη από δύο στενά συγγενικούς προγόνους φέρει υψηλό *F*.
- Η επιλογή **MATRIX** τυπώνει τον πλήρη πίνακα προσθετικής συγγένειας ώστε να διαβάζουμε απευθείας την κατά ζεύγη κληρονομιά.
- Η επιλογή **AVERAGE** προσθέτει τον μέσο συντελεστή ενδογαμίας για όλο τον στόλο — μια ενιαία σύνοψη ποικιλομορφίας σχεδίασης.

Τιμές κοντά στο 0 σημαίνουν ανεξάρτητους κλάδους· το *F* αυξάνεται όσο οι δύο πρόγονοι μιας σχεδίασης γίνονται πιο στενά συγγενικοί.

In [2]:
TITLE 'Συντελεστές Ενδογαμίας ανά Γενιά Σχεδίασης';
ΔΙΑΔΙΚΑΣΙΑ inbreed ΔΕΔΟΜΕΝΑ=DesignLineage ind average MATRIX;
   ΜΕΤΑΒΛΗΤΗ ID Parent1 Parent2;
   ΚΛΑΣΗ Generation;
ΕΚΤΕΛΕΣΗ;

                                       Συντελεστές Ενδογαμίας ανά Γενιά Σχεδίασης                                       


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coefficients (Generation 2)
--------------------------------------
ID                  F (Inbreeding)
G2_D1                 0.000000
G2_D2                 0.000000
G2_D3                 0.000000
G2_D4                 0.000000
Average Inbreeding Coefficient: 0.000000

Inbreeding Coe


NOTE: Option TITLE changed to Συντελεστές Ενδογαμίας ανά Γενιά Σχεδίασης.
NOTE: PROC INBREED data=DesignLineage



## Βήμα 3 — Συντελεστές συνδιακύμανσης αποθηκευμένοι για μετέπειτα βαθμολόγηση κινδύνου

Αντικαθιστώντας την προβολή ενδογαμίας με την επιλογή **COVAR** αναφέρονται οι ίδιες σχέσεις ως **συντελεστές συνδιακύμανσης (προσθετικής συγγένειας)**, η μορφή που μια ομάδα διαχείρισης περιουσιακών στοιχείων θα τροφοδοτούσε σε μια βαθμολογία κινδύνου εφεδρείας. Η επιλογή **OUTCOV=** αποθηκεύει τον πλήρη πίνακα ως σύνολο δεδομένων (`DesignCovar`), όπου οι στήλες `Col1`-`Col12` περιέχουν τη συγγένεια κάθε σχεδίασης με κάθε άλλη σχεδίαση (με σειρά γενεαλογικού δέντρου) — έτοιμες για σύνδεση με τις αναθέσεις υποσταθμών.

Τυπώνουμε το σύνολο δεδομένων εξόδου ώστε ο αποθηκευμένος πίνακας να είναι ορατός δίπλα στη λίστα εκτύπωσης.

In [3]:
TITLE 'Συντελεστές Συνδιακύμανσης (Συγγένειας)';
ΔΙΑΔΙΚΑΣΙΑ inbreed ΔΕΔΟΜΕΝΑ=DesignLineage covar MATRIX OUTCOV=DesignCovar;
   ΜΕΤΑΒΛΗΤΗ ID Parent1 Parent2;
   ΚΛΑΣΗ Generation;
ΕΚΤΕΛΕΣΗ;

TITLE 'Πίνακας Συγγένειας OUTCOV= Αποθηκευμένος για Βαθμολόγηση Κινδύνου';
ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=DesignCovar noobs;
ΕΚΤΕΛΕΣΗ;

                                        Συντελεστές Συνδιακύμανσης (Συγγένειας)                                         


                       The INBREED Procedure                       

Dataset: DesignLineage
Number of Observations: 12

Pedigree Summary by Generation/Class
--------------------------------------
Generation 1            Members = 4
Generation 2            Members = 4
Generation 3            Members = 4

Inbreeding Coefficients (Generation 1)
--------------------------------------
ID                  F (Inbreeding)
G1_P1                 0.000000
G1_P2                 0.000000
G1_P3                 0.000000
G1_P4                 0.000000

Covariance Matrix (Additive Genetic Relationship) (Generation 1)
--------------------------------------------------
ID                     G1_P1    G1_P2    G1_P3    G1_P4
G1_P1                1.0000   0.0000   0.0000   0.0000
G1_P2                0.0000   1.0000   0.0000   0.0000
G1_P3                0.0000   0.0000   1.0000   0.00


NOTE: Option TITLE changed to Συντελεστές Συνδιακύμανσης (Συγγένειας).
NOTE: PROC INBREED data=DesignLineage

NOTE: OUTCOV= dataset DesignCovar written.
NOTE: Option TITLE changed to Πίνακας Συγγένειας OUTCOV= Αποθηκευμένος για Βαθμολόγηση Κινδύνου.
NOTE: PROC PRINT data=DesignCovar

NOTE: PROC PRINT completed: 12 observations printed, 17 variables


## Βήμα 4 — Συνδυασμοί με τη μικρότερη συγγένεια για εφεδρικές (N-1) εγκαταστάσεις

Ο αποθηκευμένος πίνακας `DesignCovar` είναι ακριβώς αυτό που χρειάζεται μια μελέτη εφεδρείας. Η συγγένεια της σχεδίασης *i* με τη σχεδίαση *j* βρίσκεται στη γραμμή *i*, στήλη `Col`*j* (οι στήλες είναι με σειρά γενεαλογικού δέντρου, άρα τα `Col9`-`Col12` αντιστοιχούν στα τέσσερα τρέχοντα μοντέλα στόλου `G3_T1`-`G3_T4`).

Διαβάζουμε τις τέσσερις γραμμές του τρέχοντος στόλου από το `DesignCovar`, σχηματίζουμε κάθε μη διατεταγμένο ζεύγος του τρέχοντος στόλου, προσαρτούμε τον συντελεστή συγγένειάς του, και ταξινομούμε με τη μικρότερη συγγένεια πρώτα. Ο στόχος είναι να επιλέξουμε εφεδρικά ζεύγη με τη **χαμηλότερη** κοινή κληρονομιά — αυτά ελαχιστοποιούν την πιθανότητα ένα κληρονομημένο σχεδιαστικό ελάττωμα να απενεργοποιήσει και τα δύο μισά μιας προστατευμένης θέσης N-1.

In [4]:
/* Άντληση των τεσσάρων τρεχουσών γραμμών του στόλου· τα Col9..Col12 είναι
   οι συγγένειες με τα G3_T1..G3_T4 (σειρά στηλών γενεαλογικού δέντρου).
   Κατασκευή κάθε μη διατεταγμένου ζεύγους. */
ΔΕΔΟΜΕΝΑ Pairs;
   ΟΡΙΣΜΟΣ DesignCovar;
   ΟΠΟΥ ID IN ('G3_T1','G3_T2','G3_T3','G3_T4');
   LENGTH DesignA $12 DesignB $12;
   ARRAY g3{4} Col9 Col10 Col11 Col12;
   ARRAY nm{4} $12 _temporary_
      ('G3_T1','G3_T2','G3_T3','G3_T4');
   DesignA = ID;
   ΕΠΑΝΑΛΗΨΗ j = 1 ΕΩΣ 4;
      DesignB = nm{j};
      ΕΑΝ DesignA < DesignB ΤΟΤΕ ΕΠΑΝΑΛΗΨΗ;
         Relationship = g3{j};
         ΕΞΟΔΟΣ;
      ΤΕΛΟΣ;
   ΤΕΛΟΣ;
   ΚΡΑΤΗΣΗ DesignA DesignB Relationship;
ΕΚΤΕΛΕΣΗ;

ΔΙΑΔΙΚΑΣΙΑ ΤΑΞΙΝΟΜΗΣΗ ΔΕΔΟΜΕΝΑ=Pairs; ΚΑΤΑ Relationship; ΕΚΤΕΛΕΣΗ;

TITLE 'Υποψήφιοι Συνδυασμοί Εφεδρείας (N-1), με τη Μικρότερη Συγγένεια Πρώτα';
ΔΙΑΔΙΚΑΣΙΑ ΕΚΤΥΠΩΣΗ ΔΕΔΟΜΕΝΑ=Pairs noobs ΕΤΙΚΕΤΑ;
   ΜΕΤΑΒΛΗΤΗ DesignA DesignB Relationship;
   ΕΤΙΚΕΤΑ DesignA='Σχεδίαση Α' DesignB='Σχεδίαση Β' Relationship='Συντελεστής Συγγένειας';
ΕΚΤΕΛΕΣΗ;
TITLE;

                         Υποψήφιοι Συνδυασμοί Εφεδρείας (N-1), με τη Μικρότερη Συγγένεια Πρώτα                          


         Σχεδίαση Α           Σχεδίαση Β                       Συντελεστής Συγγένειας
-------------------  -------------------  -------------------------------------------
G3_T1                G3_T3                                                          0
G3_T2                G3_T4                                                       0.25
G3_T1                G3_T2                                                      0.375
G3_T1                G3_T4                                                      0.375
G3_T2                G3_T3                                                      0.375
G3_T3                G3_T4                                                      0.375




NOTE: DATA Pairs


NOTE: Read 12 rows from DesignCovar.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC SORT data=Pairs

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 6 rows from Pairs.
NOTE: Wrote Pairs (6 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: Option TITLE changed to Υποψήφιοι Συνδυασμοί Εφεδρείας (N-1), με τη Μικρότερη Συγγένεια Πρώτα.
NOTE: PROC PRINT data=Pairs

NOTE: PROC PRINT completed: 6 observations printed, 3 variables


## Ερμηνεία των αποτελεσμάτων

**Συντελεστές ενδογαμίας (Βήμα 2).** Όλες οι ιδρυτικές πλατφόρμες Γενιάς 1 και όλα τα παράγωγα Γενιάς 2 εμφανίζουν *F* = 0 — εκ κατασκευής κανένα δεν έχει δύο συγγενικούς προγόνους. Δύο μοντέλα Γενιάς 3 εμφανίζονται με *F* = 0,25: το `G3_T1` (κατασκευασμένο από το ζευγάρι αδελφών `G2_D1` x `G2_D2`) και το `G3_T3` (από το ζευγάρι αδελφών `G2_D3` x `G2_D4`). Οι πρόγονοί τους ανάγονται στο *ίδιο* ζευγάρι ιδρυτικών πλατφορμών, οπότε η κληρονομιά τους είναι συμπυκνωμένη. Από την άποψη της αξιοπιστίας, αυτές είναι οι σχεδιάσεις πιο εκτεθειμένες σε ένα μοναδικό κληρονομημένο ελάττωμα, και δικαιολογούν επιπλέον ανεξάρτητες δοκιμές πιστοποίησης. Τα `G3_T2` και `G3_T4`, που διασταυρώνουν τους δύο ανεξάρτητους κλάδους, έχουν *F* = 0.

**Πίνακας συγγένειας (Βήμα 3).** Τα εκτός διαγωνίου στοιχεία ποσοτικοποιούν την κατά ζεύγη κοινή κληρονομιά. Τα δύο ζευγάρια αδελφών Γενιάς 2 εμφανίζουν το καθένα συγγένεια 0,50 μεταξύ τους (`G2_D1`-`G2_D2` και `G2_D3`-`G2_D4`), ενώ τα παράγωγα από αντίθετους κλάδους βρίσκονται στο 0,00. Οι ενδογαμικές τρέχουσες σχεδιάσεις στόλου φέρουν αυτο-συγγένεια 1,25 (= 1 + *F*), ορατή στη διαγώνιο. Το σύνολο δεδομένων `DesignCovar` καθιστά τον πλήρη πίνακα διαθέσιμο για σύνδεση με τις αναθέσεις υποσταθμών.

**Συνδυασμοί με τη μικρότερη συγγένεια (Βήμα 4).** Κατατάσσοντας κάθε ζεύγος του τρέχοντος στόλου κατά συγγένεια, το **`G3_T1` & `G3_T3` έρχεται πρώτο με συγγένεια 0,00** — κατάγονται από εντελώς ανεξάρτητες πρόγονες πλατφόρμες και δεν μοιράζονται καμία κληρονομιά σχεδίασης. Αυτός είναι ο ισχυρότερος εφεδρικός συνδυασμός, και είναι ιδιαίτερα πολύτιμος επειδή το `G3_T1` είναι από μόνο του μία από τις δύο πιο ενδογαμικές σχεδιάσεις: ο συνδυασμός του με έναν εντελώς ασυγγενή σύντροφο αντισταθμίζει τη συμπυκνωμένη κληρονομιά του. Το επόμενο καλύτερο ζεύγος είναι το `G3_T2` & `G3_T4` με 0,25· τα υπόλοιπα ζεύγη βρίσκονται στο 0,375. Ο μέσος συντελεστής ενδογαμίας του στόλου **0,0417** (τυπωμένος από την επιλογή AVERAGE στο Βήμα 2) συνοψίζει τη συνολική ποικιλομορφία σχεδίασης. Οι προμήθειες θα πρέπει να προτιμούν τα ζεύγη με τη χαμηλότερη συγγένεια για θέσεις N-1 και, με την πάροδο του χρόνου, να διευρύνουν την προγονική βάση — το ισοδύναμο στη μηχανολογία περιουσιακών στοιχείων της αποφυγής μιας γενετικής στένωσης (bottleneck).

**Επιφύλαξη.** Πρόκειται για ενδεικτικά συνθετικά δεδομένα· μια μελέτη παραγωγής θα έχτιζε το γενεαλογικό δέντρο από τα πραγματικά αρχεία αναθεώρησης σχεδίασης του κατασκευαστή και θα επικύρωνε τους βαθμούς συγγένειας έναντι ιστορικών συμβάντων διακοπής κοινής αιτίας πριν καθοδηγήσει αποφάσεις προμηθειών.